# Nook 3DGS Pipeline — Kaggle GPU

**Before running:**
1. Create a Kaggle dataset with your video: kaggle.com/datasets → New Dataset → upload your MOV/MP4
2. In this notebook: Add Data (top right) → Your Datasets → select your video dataset
3. Session options (top right) → Accelerator → **GPU T4 x2** (or P100)
4. Run all cells top to bottom

Your video will be at `/kaggle/input/<dataset-name>/<filename>`

## 0. Confirm GPU + find video

In [ ]:
import subprocess, os, glob, shutil

if not shutil.which('nvidia-smi'):
    raise RuntimeError('No GPU. Session options → Accelerator → GPU T4 x2')
r = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap,memory.total', '--format=csv'],
                   capture_output=True, text=True)
print(r.stdout)

# Auto-find the video in /kaggle/input
videos = glob.glob('/kaggle/input/**/*.mov', recursive=True) + \
         glob.glob('/kaggle/input/**/*.MOV', recursive=True) + \
         glob.glob('/kaggle/input/**/*.mp4', recursive=True) + \
         glob.glob('/kaggle/input/**/*.MP4', recursive=True)
if not videos:
    raise FileNotFoundError(
        'No video found in /kaggle/input.\n'
        'Add your dataset: top-right Add Data → Your Datasets'
    )
VIDEO = videos[0]
print(f'Video: {VIDEO}  ({os.path.getsize(VIDEO)/1e6:.0f} MB)')

## 1. Install COLMAP + nerfstudio + pre-compile gsplat

~20–25 min. gsplat compiles its CUDA kernels here (MAX_JOBS=1 to avoid OOM). Done once, cached for the whole session.

In [ ]:
import subprocess, os

# Stream output live so you can see progress instead of staring at silence
print('Installing COLMAP + ffmpeg (10-20 min, many dependencies)...')
subprocess.run('apt-get install -y colmap ffmpeg 2>&1 | tail -5', shell=True)

print('\nInstalling nerfstudio + gsplat...')
subprocess.run('pip install -q ninja nerfstudio gsplat 2>&1 | tail -5', shell=True)

import importlib.metadata as _md
import torch
print(f'\ntorch {torch.__version__} | cuda {torch.version.cuda} | {torch.cuda.get_device_name(0)}')
print('nerfstudio', _md.version('nerfstudio'), '| gsplat', _md.version('gsplat'))

# Pre-compile gsplat CUDA kernels now — MAX_JOBS=1 prevents OOM during parallel nvcc
print('\nPre-compiling gsplat CUDA kernels (15-20 min)...')
print('You will see [1/26], [2/26] etc. ticking — that is normal.')
env = {**os.environ, 'MAX_JOBS': '1'}
r = subprocess.run(
    ['python', '-c',
     'import os; os.environ["MAX_JOBS"]="1"; '
     'import torch; from gsplat import rasterization; '
     'print("gsplat compiled OK")'],
    env=env, text=True
)
if r.returncode != 0:
    raise RuntimeError('gsplat compile failed')
print('All done.')

## 2. Extract 100 frames

Reads the Kaggle dataset video directly (already on local disk — no Drive copy needed). Extracts 100 evenly-spaced frames at 960×540. Skips re-encoding entirely.

In [ ]:
import subprocess, os, glob, time

FRAMES = '/kaggle/working/frames'
os.makedirs(FRAMES, exist_ok=True)

# Probe duration
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
     '-show_entries', 'stream=width,height', '-show_entries', 'format=duration',
     '-of', 'default=noprint_wrappers=1', VIDEO],
    capture_output=True, text=True
)
print('Source:', probe.stdout.replace('\n', '  '))

duration = next((l.split('=')[1] for l in probe.stdout.split('\n') if l.startswith('duration')), None)
fps_target = 100 / float(duration) if duration else 0.5

print(f'Extracting 100 frames at {fps_target:.3f}fps...')
t0 = time.time()
r = subprocess.run([
    'ffmpeg', '-y', '-i', VIDEO,
    '-vf', f'fps={fps_target:.4f},scale=960:540:force_original_aspect_ratio=decrease,scale=trunc(iw/2)*2:trunc(ih/2)*2',
    '-q:v', '2',
    f'{FRAMES}/frame_%04d.jpg'
], capture_output=True, text=True)

if r.returncode != 0:
    print(r.stderr[-300:]); raise RuntimeError('ffmpeg failed')

n = len(glob.glob(f'{FRAMES}/*.jpg'))
print(f'Done in {time.time()-t0:.0f}s — {n} frames')

## 3. COLMAP structure-from-motion

CPU SIFT with sequential matching (correct for video, O(n) not O(n²)). ~5 min for 100 frames.

In [ ]:
import subprocess, os

DB     = '/kaggle/working/colmap/database.db'
SPARSE = '/kaggle/working/colmap/sparse'
DATA   = '/kaggle/working/data'

os.makedirs(os.path.dirname(DB), exist_ok=True)
os.makedirs(SPARSE, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

print('1/4 Feature extraction...')
subprocess.run(['colmap', 'feature_extractor',
    '--database_path', DB, '--image_path', FRAMES,
    '--ImageReader.single_camera', '1',
    '--ImageReader.camera_model', 'OPENCV',
    '--SiftExtraction.use_gpu', '0'], check=True)

print('\n2/4 Sequential matching...')
subprocess.run(['colmap', 'sequential_matcher',
    '--database_path', DB,
    '--SiftMatching.use_gpu', '0',
    '--SequentialMatching.overlap', '10'], check=True)

print('\n3/4 Sparse reconstruction...')
subprocess.run(['colmap', 'mapper',
    '--database_path', DB, '--image_path', FRAMES,
    '--output_path', SPARSE], check=True)

models = sorted(os.listdir(SPARSE))
if not models: raise RuntimeError('COLMAP produced no sparse model')
print(f'Models: {models}')

print('\n4/4 Converting to nerfstudio format...')
r = subprocess.run(['ns-process-data', 'images',
    '--data', FRAMES, '--output-dir', DATA,
    '--skip-colmap', '--colmap-model-path', f'{SPARSE}/{models[0]}',
    '--verbose'], text=True)
if r.returncode != 0: raise RuntimeError('ns-process-data failed')
print('Done.')

## 4. Train splatfacto

~10–15 min on T4. gsplat is already compiled from step 1 so training starts immediately.

In [ ]:
import subprocess, os

SPLAT = '/kaggle/working/splat'
env   = {**os.environ, 'MAX_JOBS': '1'}

r = subprocess.run([
    'ns-train', 'splatfacto',
    '--data', DATA,
    '--output-dir', SPLAT,
    '--max-num-iterations', '3000',
    '--viewer.quit-on-train-completion', 'True',
    'nerfstudio-data',
    '--downscale-factor', '2',
], text=True, env=env)

if r.returncode != 0:
    raise RuntimeError(f'ns-train failed (exit {r.returncode})')
print('Training complete.')

## 5. Export PLY

In [ ]:
import subprocess, os, glob

configs = sorted(glob.glob(f'{SPLAT}/**/config.yml', recursive=True))
if not configs: raise RuntimeError('No config.yml — did training complete?')
config = configs[-1]
print('Config:', config)

OUT = '/kaggle/working/output'
os.makedirs(OUT, exist_ok=True)

r = subprocess.run(['ns-export', 'gaussian-splat',
    '--load-config', config, '--output-dir', OUT], text=True)
if r.returncode != 0: raise RuntimeError('ns-export failed')

plys = sorted(glob.glob(f'{OUT}/**/*.ply', recursive=True))
if not plys: raise RuntimeError('No PLY file found after export')
PLY = plys[-1]
print(f'PLY: {PLY}  ({os.path.getsize(PLY)/1e6:.1f} MB)')

## 6. Download PLY + view in SuperSplat

In Kaggle: Files panel (right sidebar) → navigate to `/kaggle/working/output/` → download `splat.ply`. Then drop it into **https://superspl.at/editor** to view.

In [ ]:
print(f'PLY is at: {PLY}')
print('Download it from the Files panel on the right sidebar.')
print('Then drop it into: https://superspl.at/editor')

## 7. (Optional) Upload PLY to Vercel Blob + wire into app

Set `BLOB_TOKEN` to push straight to production. Then in Supabase dashboard set the tour's `ply_url` and `status = complete` to test the real viewer.

In [ ]:
import requests, os

BLOB_TOKEN = ''  # paste BLOB_READ_WRITE_TOKEN here
TOUR_ID    = 'ff6e9a13-99c9-4d99-9e33-db5a62afe8af'

if BLOB_TOKEN:
    remote = f'tours/{TOUR_ID}/splat.ply'
    print(f'Uploading {os.path.getsize(PLY)/1e6:.1f} MB...')
    with open(PLY, 'rb') as f:
        r = requests.put(
            f'https://blob.vercel-storage.com/{remote}',
            headers={'Authorization': f'Bearer {BLOB_TOKEN}',
                     'x-content-type': 'model/vnd.ply',
                     'x-cache-control-max-age': '31536000'},
            data=f, timeout=600
        )
    r.raise_for_status()
    url = r.json()['url']
    print('Uploaded:', url)
    print('SuperSplat:', f'https://superspl.at/editor?load={url}')
    print(f'\nSupabase: set ply_url={url} and status=complete on tour {TOUR_ID}')
else:
    print('Set BLOB_TOKEN above to enable upload.')